In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, MllamaForConditionalGeneration, AutoProcessor
# from transformers import BitsAndBytesConfig
from transformers import Gemma3ForCausalLM
from neural_controllers import NeuralController
from utils import LLMType
from collections import namedtuple 
from tqdm import tqdm 
import gc
import os

In [3]:
# from janus.utils.io import load_pil_images
# from generation_utils import extract_image
import utils

In [4]:
SEED = 0
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
np.random.seed(SEED)

In [5]:
LLM = namedtuple('LLM', ['language_model', 'tokenizer', 'processor', 'name', 'model_type'])


def select_llm(model_type, MODEL_VERSION='3.1', MODEL_SIZE='8B'):

    custom_cache_dir = "/scratch/bbjr/skarmakar/huggingface"

    if model_type=='llama':

        if MODEL_VERSION == '3.1' and MODEL_SIZE == '8B':
            model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
        elif MODEL_VERSION == '3.1' and MODEL_SIZE == '70B':
            model_id = "unsloth/Meta-Llama-3.1-70B-Instruct-bnb-4bit"
        elif MODEL_VERSION == '3.3' and MODEL_SIZE == '70B':
            model_id = "unsloth/Llama-3.3-70B-Instruct-bnb-4bit"

        language_model = AutoModelForCausalLM.from_pretrained(
            model_id, device_map="cuda", cache_dir=custom_cache_dir,
        )

        use_fast_tokenizer = "LlamaForCausalLM" not in language_model.config.architectures
        tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=use_fast_tokenizer, padding_side="left", legacy=False)
        tokenizer.pad_token_id = 0 
        # model_name='llama_3_8b_it'
        if MODEL_VERSION == '3.1' and MODEL_SIZE == '8B':
            model_name='llama_3_8b_it_eng_only'
        elif MODEL_VERSION == '3.1' and MODEL_SIZE == '70B':
            model_name = "llama_3.1_70b_it_eng_only"
        elif MODEL_VERSION == '3.3' and MODEL_SIZE == '70B':
            model_name = "llama_3.3_70b_it_eng_only"

        processor = None
        llm_type = LLMType.TEXT

        language_model.generation_config.pad_token_id = tokenizer.pad_token_id # to disable the warning

        language_model.generation_config.temperature=None # to disable the stupid warnings
        language_model.generation_config.top_p=None # to disable the stupid warnings
        # language_model.generation_config.top_k=None # to disable the stupid warnings

    elif model_type=='gemma':
        # quantization_config = BitsAndBytesConfig(load_in_8bit=True)
        if MODEL_VERSION == '3.1' and MODEL_SIZE == '1B':
            model_id = "google/gemma-3-1b-it"

        tokenizer = AutoTokenizer.from_pretrained(model_id)

        # language_model = Gemma3ForCausalLM.from_pretrained(
        #     model_id, quantization_config=quantization_config
        # ).eval()

        language_model = Gemma3ForCausalLM.from_pretrained(
            model_id, torch_dtype=torch.float16, cache_dir=custom_cache_dir,
        ).eval()
        

        if MODEL_VERSION == '3.1' and MODEL_SIZE == '1B':
            model_name='gemma_3_1b_it_eng_only'

        processor = None
        llm_type = LLMType.GEMMA_TEXT

        # print(tokenizer.chat_template)

    llm = LLM(language_model, tokenizer, processor, model_name, llm_type)
    # print(llm.language_model)
    return llm


def compute_save_directions_translate(llm, dataset, concept, control_method='rfm', orig_lang="", path='directions/'):

    if os.path.exists(os.path.join(path, f'{control_method}_{orig_lang}TO{concept}_{llm.name}.pkl')):
        print(f"'{os.path.join(path, f'{control_method}_{orig_lang}TO{concept}_{llm.name}.pkl')}' exists, skipping it.")
        return

    
    concept_types = [concept]
    for concept_type in concept_types:
        controller = NeuralController(
            llm,
            llm.tokenizer,
            rfm_iters=8,
            control_method=control_method,
            n_components=1,
        )
        controller.compute_directions(dataset[concept_type]['train']['inputs'], dataset[concept_type]['train']['labels'])

        controller.save_translate(concept=f'{concept_type}', model_name=llm.name, orig_lang=orig_lang, path=path)        



def read_file(fname, lower=True):

    concepts = []
    with open(fname, encoding="utf-8") as f: 
        for line in f:
            if lower:
                concepts.append(line.strip().lower())
            else:
                concepts.append(line.strip())
    concepts = sorted(list(set(concepts)))
    return concepts

In [6]:
torch.backends.cudnn.benchmark = True 
torch.backends.cuda.matmul.allow_tf32 = True

In [7]:
# model_type = 'llama'
model_type = 'gemma'

MODEL_VERSION='3.1'

# MODEL_SIZE='8B'
MODEL_SIZE='1B'

llm = select_llm(model_type, MODEL_VERSION=MODEL_VERSION, MODEL_SIZE=MODEL_SIZE)

In [8]:
hidden_layers = list(range(-1, -llm.language_model.config.num_hidden_layers, -1))
print(hidden_layers)

[-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25]


In [ ]:
original_langs = ['english', 'french', 'german', 'hindi', 'italian', 'portuguese', 'spanish', 'thai']
other_langs = ['english', 'french', 'german', 'hindi', 'italian', 'portuguese', 'spanish', 'thai']

# dataset_label = 'language'
dataset_label = 'language_multi'

# save_path = 'all_gitignore/language/'
# save_path = 'all_gitignore/language_multi/'


save_path = 'all_gitignore/language_multi_gemma/'


# METHOD = 'mean_difference'
METHOD = 'rfm'

for original_lang in original_langs:
    for other_lang in other_langs:
        if original_lang == other_lang:
            continue
        
        print(f"============================================={original_lang} to {other_lang}=============================================")
        
        if dataset_label == 'language':
            dataset = utils.language_dataset(llm, original_lang, other_lang)
        elif dataset_label == 'language_multi':
            dataset = utils.multilingual_dataset(llm, original_lang, other_lang)


        compute_save_directions_translate(llm, dataset, other_lang, control_method=METHOD, orig_lang=original_lang, path=save_path)
        # del llm
        del dataset
        torch.cuda.empty_cache()

        gc.collect()

=============================================english to french=============================================
Loading FLORES-200 for english (eng_Latn) and french (fra_Latn)...
train 400
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

use_concat False


  0%|          | 0/25 [00:00<?, ?it/s]